# Exemplo de uso: `BaseDataProcessorSpark` + `DataDictionaryBuilder`

Fluxo completo de ponta a ponta:

1. gera um CSV de exemplo "sujo" em um Volume;
2. lê, limpa e transforma com `BaseDataProcessorSpark`;
3. exporta para uma tabela Delta com metadados de governança;
4. monta o dicionário de dados com `DataDictionaryBuilder` (incluindo descrições por IA).

**Pré-requisito:** compute serverless com o base environment `dataeng-codes` selecionado no painel *Environment*
(veja `DEPLOY.md`). Não é preciso `%pip install` nem `sys.path`.

> Este notebook fica na mesma pasta dos `.py`, então aqui ele usa o **código local** (modo desenvolvimento).
> Copiado para qualquer outra pasta, ele usa o **pacote publicado**. A célula de verificação abaixo mostra qual está em uso.

In [0]:
# Recarrega os módulos automaticamente quando os .py mudam (útil no modo desenvolvimento)
%load_ext autoreload
%autoreload 2

## 1. Imports e verificação da versão

In [0]:
from importlib.metadata import version, PackageNotFoundError

import BaseDataProcessorSpark as bdp_mod
from BaseDataProcessorSpark import BaseDataProcessorSpark
from DataDictionaryBuilder import DataDictionaryBuilder

try:
    print("Pacote instalado: dataeng-codes", version("dataeng-codes"))
except PackageNotFoundError:
    print("Pacote dataeng-codes não instalado (selecione o base environment 'dataeng-codes').")

origem = "código local da Git folder" if "/site-packages/" not in bdp_mod.__file__ else "pacote publicado (wheel)"
print("Módulo em uso:", bdp_mod.__file__, "->", origem)

## 2. Configuração

Ajuste catálogo, schema e volume. O buffer de governança precisa ser o **mesmo caminho**
no `BaseDataProcessorSpark` (`buffer_log_path`) e no `DataDictionaryBuilder` (`buffer_path`).

In [0]:
CATALOG = "workspace"          # catálogo onde ficam as tabelas
SCHEMA = "exemplo_dataeng"     # schema de destino (criado se não existir)
VOLUME = "landing"             # volume para arquivos de entrada/saída

BASE_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
INPUT_DIR = f"{BASE_VOLUME}/entrada"
OUTPUT_DIR = f"{BASE_VOLUME}/saida"
BUFFER_PATH = f"{OUTPUT_DIR}/export_log_buffer_.txt"

TABELA_DADOS = "vendas_exemplo"
TABELA_GOVERNANCA = "data_dictionary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Entrada:", INPUT_DIR)
print("Saída:  ", OUTPUT_DIR)

## 3. Gerar um arquivo de exemplo

CSV com problemas comuns de ingestão: nomes de coluna com acento, espaço e símbolos,
uma linha totalmente vazia, uma linha duplicada e valores nulos.

In [0]:
import pandas as pd

amostra = pd.DataFrame(
    [
        ["2026-01-05", "Sul",     "Tubo PVC 50mm", 120, 18.90, "Ana"],
        ["2026-01-05", "Sul",     "Tubo PVC 50mm", 120, 18.90, "Ana"],    # duplicada
        ["2026-01-07", "Sudeste", "Conexão T",      40,  6.50, None],     # vendedor nulo
        ["2026-01-09", "Nordeste", "Registro 3/4",  None, 32.00, "Bruno"], # quantidade nula
        [None, None, None, None, None, None],                              # linha vazia
        ["2026-02-02", "SUL",     "Joelho 90",      75,  4.20, "Carla"],   # região fora do padrão
        ["2026-02-10", "Norte",   "Tubo PVC 50mm",  60, 19.40, "Davi"],
    ],
    columns=["Data da Venda", "Região", "Produto", "Quantidade", "Valor Unitário (R$)", "Vendedor"],
)
arquivo = f"{INPUT_DIR}/vendas_exemplo.csv"
amostra.to_csv(arquivo, sep=";", index=False)
print("Arquivo gerado:", arquivo)

## 4. Ingestão com `BaseDataProcessorSpark`

In [0]:
processor = BaseDataProcessorSpark(
    input_directory=INPUT_DIR,
    output_directory=OUTPUT_DIR,
    catalog=CATALOG,
    buffer_log_path=BUFFER_PATH,
)

print("Arquivos disponíveis:", processor.list_files())
df = processor.read_data("vendas_exemplo.csv", separator=";")
df.printSchema()
display(df)

## 5. Limpeza e transformação

In [0]:
# Nomes de coluna: "Valor Unitário (R$)" -> "valor_unitario_r", "Região" -> "regiao" etc.
df = processor.sanitize_column_names(df)

# Remove a linha vazia e a duplicada
df = processor.drop_null_rows(df)
df = processor.drop_duplicates(df)

# Tipos corretos
df = processor.change_data_types(df, {
    "data_da_venda": "date",
    "quantidade": "int",
    "valor_unitario_r": "double",
})

# Padroniza valores e preenche nulos de acordo com o tipo (0 para números, "Não identificado" para texto)
df = processor.tratar_colunas(df, ["regiao"], {"SUL": "Sul"})
df = processor.fill_nan_based_on_dtype(df)

# Nova coluna calculada
from pyspark.sql import functions as F
df = df.withColumn("valor_total", F.round(F.col("quantidade") * F.col("valor_unitario_r"), 2))

df = processor.rename_columns(df, {"valor_unitario_r": "valor_unitario"})
df = processor.ordenar_dataframe(df, ["data_da_venda"], [True])
display(df)

## 6. Consultas SQL sobre o DataFrame

In [0]:
processor.set_df(df)
processor.create_temp_view("vw_vendas")

resumo = processor.get_sql_query("""
    SELECT regiao,
           COUNT(*)          AS qtd_vendas,
           SUM(quantidade)   AS itens,
           SUM(valor_total)  AS faturamento
    FROM vw_vendas
    GROUP BY regiao
    ORDER BY faturamento DESC
""")

## 7. Exportação com governança

`export_to_db` grava a tabela Delta e, junto:
- adiciona `write_date` / `write_time`;
- grava `TBLPROPERTIES` `governanca.*` (notebook, usuário, data e arquivos de origem);
- aplica o comentário da tabela (`descricao`);
- acrescenta uma linha por coluna no buffer de governança.

In [0]:
processor.set_df(df)
tabela = processor.export_to_db(
    database=SCHEMA,
    table=TABELA_DADOS,
    mode="overwrite",
    descricao="Vendas de exemplo por região e produto, geradas pelo notebook Class_usage_example.",
)
print("Tabela exportada:", tabela)

In [0]:
# Metadados gravados na tabela
display(spark.sql(f"SHOW TBLPROPERTIES `{CATALOG}`.`{SCHEMA}`.`{TABELA_DADOS}`")
        .filter("key LIKE 'governanca.%'"))

# Conteúdo do buffer que o DataDictionaryBuilder vai consumir
print(open(BUFFER_PATH, encoding="utf-8").read())

## 8. Dicionário de dados com `DataDictionaryBuilder`

`run_full_pipeline` executa, em ordem: criação da tabela de governança → carga do buffer (MERGE) →
descrições por IA para tabelas/colunas sem comentário → sincronização das descrições no dicionário.

Se nenhum modelo de IA estiver disponível, a etapa de IA é pulada com aviso e o restante segue normalmente.
`limite_itens` limita as chamadas de IA (útil para testar).

In [0]:
builder = DataDictionaryBuilder(
    target_catalog=CATALOG,
    target_schema=SCHEMA,
    target_table=TABELA_GOVERNANCA,
    buffer_path=BUFFER_PATH,
)

builder.run_full_pipeline(gerar_descricoes_ia=True, limite_itens=20)

In [0]:
display(spark.sql(f"""
    SELECT `table`, `column`, collumn_dtype, column_description, table_description,
           `user`, file_fetched, source_notebook_url, creation_time, last_update_time
    FROM `{CATALOG}`.`{SCHEMA}`.`{TABELA_GOVERNANCA}`
    WHERE `table` = '{TABELA_DADOS}'
    ORDER BY `column`
"""))

## 9. Limpeza (opcional)

Descomente para remover tudo o que este exemplo criou.

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS `{CATALOG}`.`{SCHEMA}`.`{TABELA_DADOS}`")
# spark.sql(f"DROP TABLE IF EXISTS `{CATALOG}`.`{SCHEMA}`.`{TABELA_GOVERNANCA}`")
# spark.sql(f"DROP VOLUME IF EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")